Nakatsukasa

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import random

# コードの最初に挿入
torch.manual_seed(42)
np.random.seed(42)

# GPUパソコンか否かの判定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")

# 1. Franke関数の定義
def franke_function(x, y):
    term1 = 0.75 * np.exp(-(9*x - 2)**2 / 4 - (9*y - 2)**2 / 4)
    term2 = 0.75 * np.exp(-(9*x + 1)**2 / 49 - (9*y + 1)**2 / 10)
    term3 = 0.50 * np.exp(-(9*x - 7)**2 / 4 - (9*y - 3)**2 / 4)
    term4 = -0.2 * np.exp(-(9*x - 4)**2 - (9*y - 7)**2)
    return term1 + term2 + term3 + term4

f = franke_function

# 2. データ生成
grid_size = 50
x_coord = np.linspace(0, 1, grid_size)
y_coord = np.linspace(0, 1, grid_size)
X_mesh, Y_mesh = np.meshgrid(x_coord, y_coord)
# GPUパソコンなら .to(device)
features = torch.tensor(np.stack([X_mesh.ravel(), Y_mesh.ravel()], axis=1), dtype=torch.float32).to(device)
labels = torch.tensor(f(X_mesh.ravel(), Y_mesh.ravel()), dtype=torch.float32).view(-1, 1).to(device)

# ==========================================
# 活性化関数とモデル定義
# ==========================================
class RationalActivation(nn.Module):
    def __init__(self, mode='nakatsukasa'):
        super().__init__()
        self.mode = mode
        
        if mode == 'nakatsukasa':
            # Nakatsukasa方式: (3, 2)型
            # ReLUの挙動に近づけるための初期化
            self.a = nn.Parameter(torch.tensor([0.0, 0.5, 0.5, 0.0], dtype=torch.float32))
            self.b = nn.Parameter(torch.tensor([0.0, 1.0], dtype=torch.float32))
            
        elif mode == 'molina':
            # Molina方式: (5, 4)型
            # Leaky ReLU (alpha=0.01) をパデ近似する厳密な初期化係数
            self.a = nn.Parameter(torch.tensor([0.02979246, 0.61837738, 2.32335207, 3.05202660, 1.48548002, 0.25103717], dtype=torch.float32))
            self.b = nn.Parameter(torch.tensor([1.14201226, 4.39322834, 0.87154450, 0.34720652], dtype=torch.float32))
            
        else:
            raise ValueError("mode must be 'nakatsukasa' or 'molina'")

    def forward(self, x):
        # 分子 P(x) の計算 (a_0 + a_1*x + a_2*x^2 + ...)
        numerator = self.a[0]
        for i in range(1, len(self.a)):
            numerator = numerator + self.a[i] * (x ** i)
        
        # 分母 Q(x) の変数部分の計算 (b_1*x + b_2*x^2 + ...)
        denominator_inner = 0.0
        for i in range(len(self.b)):
            denominator_inner = denominator_inner + self.b[i] * (x ** (i + 1))
            
        # 安定性のための絶対値バリア (MolinaのSafe PAU構造)[cite: 5]
        denominator = 1.0 + torch.abs(denominator_inner)
        
        return numerator / (denominator + 1e-6)

# ==========================================
# 3. モデル定義（残差接続 & バッチ正規化 搭載版）
# ==========================================

class RationalResBlock(nn.Module):
    """
    残差接続とバッチ正規化を備えた、独立した1つの「残差ブロック」
    入力 x に対して、 F(x) + x を計算して出力します。
    """
    def __init__(self, num_features=64, activation_mode='nakatsukasa'):
        super().__init__()
        # 1つ目のサブブロック
        self.fc1 = nn.Linear(num_features, num_features)
        self.bn1 = nn.BatchNorm1d(num_features)  # バッチ正規化
        self.act1 = RationalActivation(mode=activation_mode)
        
        # 2つ目のサブブロック
        self.fc2 = nn.Linear(num_features, num_features)
        self.bn2 = nn.BatchNorm1d(num_features)  # バッチ正規化
        self.act2 = RationalActivation(mode=activation_mode)

    def forward(self, x):
        # 残差（スキップ先）を保存
        residual = x
        
        # F(x) の計算
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.act1(out)
        
        out = self.fc2(out)
        out = self.bn2(out)
        
        # 残差接続：元の入力 x (residual) を足し合わせる
        out = out + residual
        
        # 足し算のあとに最後の活性化関数を通す
        out = self.act2(out)
        return out


class Rational_Deep_ResNet(nn.Module):
    def __init__(self, num_blocks=10,width=64, activation_mode='nakatsukasa'):
        """
        num_blocks: 積み重ねる残差ブロックの数。
        1ブロックあたり「線形層が2つ」あるため、10ブロックで約20層（Linear 20層分）の深さになります。
        """
        super().__init__()
        
        # 1. 入力層 (2次元の座標入力を 64次元に拡張)
        self.input_layer = nn.Sequential(
            nn.Linear(2, width),
            nn.BatchNorm1d(width),
            RationalActivation(mode=activation_mode)
        )
        
        # 2. 残差ブロックの積み重ね (num_blocks 個分を配置)
        blocks = []
        for _ in range(num_blocks):
            blocks.append(RationalResBlock(num_features=width, activation_mode=activation_mode))
        self.res_blocks = nn.Sequential(*blocks)
        
        # 3. 出力層 (64次元を1次元にマッピング)
        self.output_layer = nn.Linear(width, 1)
        
    def forward(self, x):
        out = self.input_layer(x)
        out = self.res_blocks(out)
        out = self.output_layer(out)
        return out


num_blocks = 2
width = 64
mode = "nakatsukasa"
num_epoch = 3000
lr = 0.005
# 実行時にブロック数を指定（10ブロック ≒ 約20層）
model = Rational_Deep_ResNet(num_blocks=num_blocks ,width=width, activation_mode=mode).to(device)
# ※必ずoptimizerを定義する「前」に .to(device) を適用してください。
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()

# 訓練
print("Rational activationモデルの訓練を開始します...")
print(f"ブロック数:{num_blocks}≒{num_blocks * 2}層, 幅:{width}, mode:{mode} ,エポック数:{num_epoch}, 学習率:{lr}")
for epoch in range(num_epoch):
    optimizer.zero_grad()
    pred = model(features)
    loss = criterion(pred, labels)
    loss.backward()
    optimizer.step()
print("訓練が完了しました。")

# 4. 近似用ラッパー関数
def f_approx(x, y):
    # モデルと同じデバイス（GPU）に入力を送る
    coords = torch.tensor(np.stack([np.array(x).ravel(), np.array(y).ravel()], axis=1), dtype=torch.float32).to(device)
    with torch.no_grad():
        # 推論結果を .cpu() で一度CPUに戻してからNumPy配列に変換する
        res = model(coords).cpu().numpy().reshape(np.shape(x))
    return res

# 5. 誤差解析
n_eval = 51
x_eval = np.linspace(0, 1, n_eval)
y_eval = np.linspace(0, 1, n_eval)
X_eval, Y_eval = np.meshgrid(x_eval, y_eval, indexing='ij')
f_eval = f(X_eval, Y_eval)
f_eval_approx = f_approx(X_eval, Y_eval)

abs_err = np.abs(f_eval - f_eval_approx)
rel_err = abs_err / np.max(np.abs(f_eval))
print(f"Max Relative Error: {np.max(rel_err):.6f}")

# 6. 可視化
fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X_eval, Y_eval, f_eval, cmap='viridis', alpha=0.8)
ax1.set_title("True Function")

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(X_eval, Y_eval, f_eval_approx, cmap='plasma', alpha=0.8)
ax2.set_title("NN Approximated (with Rational)")

fig_err = plt.figure(figsize=(7, 6))
ax3 = fig_err.add_subplot(1, 1, 1, projection='3d')
ax3.plot_surface(X_eval, Y_eval, rel_err, cmap='inferno', alpha=0.8)
ax3.set_title("Relative Error")

plt.show()

Molina

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import random

# コードの最初に挿入
torch.manual_seed(42)
np.random.seed(42)

# GPUパソコンか否かの判定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")

# 1. Franke関数の定義
def franke_function(x, y):
    term1 = 0.75 * np.exp(-(9*x - 2)**2 / 4 - (9*y - 2)**2 / 4)
    term2 = 0.75 * np.exp(-(9*x + 1)**2 / 49 - (9*y + 1)**2 / 10)
    term3 = 0.50 * np.exp(-(9*x - 7)**2 / 4 - (9*y - 3)**2 / 4)
    term4 = -0.2 * np.exp(-(9*x - 4)**2 - (9*y - 7)**2)
    return term1 + term2 + term3 + term4

f = franke_function

# 2. データ生成
grid_size = 50
x_coord = np.linspace(0, 1, grid_size)
y_coord = np.linspace(0, 1, grid_size)
X_mesh, Y_mesh = np.meshgrid(x_coord, y_coord)
# GPUパソコンなら .to(device)
features = torch.tensor(np.stack([X_mesh.ravel(), Y_mesh.ravel()], axis=1), dtype=torch.float32).to(device)
labels = torch.tensor(f(X_mesh.ravel(), Y_mesh.ravel()), dtype=torch.float32).view(-1, 1).to(device)

# ==========================================
# 活性化関数とモデル定義
# ==========================================
class RationalActivation(nn.Module):
    def __init__(self, mode='nakatsukasa'):
        super().__init__()
        self.mode = mode
        
        if mode == 'nakatsukasa':
            # Nakatsukasa方式: (3, 2)型
            # ReLUの挙動に近づけるための初期化
            self.a = nn.Parameter(torch.tensor([0.0, 0.5, 0.5, 0.0], dtype=torch.float32))
            self.b = nn.Parameter(torch.tensor([0.0, 1.0], dtype=torch.float32))
            
        elif mode == 'molina':
            # Molina方式: (5, 4)型
            # Leaky ReLU (alpha=0.01) をパデ近似する厳密な初期化係数
            self.a = nn.Parameter(torch.tensor([0.02979246, 0.61837738, 2.32335207, 3.05202660, 1.48548002, 0.25103717], dtype=torch.float32))
            self.b = nn.Parameter(torch.tensor([1.14201226, 4.39322834, 0.87154450, 0.34720652], dtype=torch.float32))
            
        else:
            raise ValueError("mode must be 'nakatsukasa' or 'molina'")

    def forward(self, x):
        # 分子 P(x) の計算 (a_0 + a_1*x + a_2*x^2 + ...)
        numerator = self.a[0]
        for i in range(1, len(self.a)):
            numerator = numerator + self.a[i] * (x ** i)
        
        # 分母 Q(x) の変数部分の計算 (b_1*x + b_2*x^2 + ...)
        denominator_inner = 0.0
        for i in range(len(self.b)):
            denominator_inner = denominator_inner + self.b[i] * (x ** (i + 1))
            
        # 安定性のための絶対値バリア (MolinaのSafe PAU構造)[cite: 5]
        denominator = 1.0 + torch.abs(denominator_inner)
        
        return numerator / (denominator + 1e-6)

# ==========================================
# 3. モデル定義（残差接続 & バッチ正規化 搭載版）
# ==========================================

class RationalResBlock(nn.Module):
    """
    残差接続とバッチ正規化を備えた、独立した1つの「残差ブロック」
    入力 x に対して、 F(x) + x を計算して出力します。
    """
    def __init__(self, num_features=64, activation_mode='nakatsukasa'):
        super().__init__()
        # 1つ目のサブブロック
        self.fc1 = nn.Linear(num_features, num_features)
        self.bn1 = nn.BatchNorm1d(num_features)  # バッチ正規化
        self.act1 = RationalActivation(mode=activation_mode)
        
        # 2つ目のサブブロック
        self.fc2 = nn.Linear(num_features, num_features)
        self.bn2 = nn.BatchNorm1d(num_features)  # バッチ正規化
        self.act2 = RationalActivation(mode=activation_mode)

    def forward(self, x):
        # 残差（スキップ先）を保存
        residual = x
        
        # F(x) の計算
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.act1(out)
        
        out = self.fc2(out)
        out = self.bn2(out)
        
        # 残差接続：元の入力 x (residual) を足し合わせる
        out = out + residual
        
        # 足し算のあとに最後の活性化関数を通す
        out = self.act2(out)
        return out


class Rational_Deep_ResNet(nn.Module):
    def __init__(self, num_blocks=10,width=64, activation_mode='nakatsukasa'):
        """
        num_blocks: 積み重ねる残差ブロックの数。
        1ブロックあたり「線形層が2つ」あるため、10ブロックで約20層（Linear 20層分）の深さになります。
        """
        super().__init__()
        
        # 1. 入力層 (2次元の座標入力を 64次元に拡張)
        self.input_layer = nn.Sequential(
            nn.Linear(2, width),
            nn.BatchNorm1d(width),
            RationalActivation(mode=activation_mode)
        )
        
        # 2. 残差ブロックの積み重ね (num_blocks 個分を配置)
        blocks = []
        for _ in range(num_blocks):
            blocks.append(RationalResBlock(num_features=width, activation_mode=activation_mode))
        self.res_blocks = nn.Sequential(*blocks)
        
        # 3. 出力層 (64次元を1次元にマッピング)
        self.output_layer = nn.Linear(width, 1)
        
    def forward(self, x):
        out = self.input_layer(x)
        out = self.res_blocks(out)
        out = self.output_layer(out)
        return out


num_blocks = 2
width = 64
mode = "molina"
num_epoch = 3000
lr = 0.005
# 実行時にブロック数を指定（10ブロック ≒ 約20層）
model = Rational_Deep_ResNet(num_blocks=num_blocks ,width=width, activation_mode=mode).to(device)
# ※必ずoptimizerを定義する「前」に .to(device) を適用してください。
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()

# 訓練
print("Rational activationモデルの訓練を開始します...")
print(f"ブロック数:{num_blocks}≒{num_blocks * 2}層, 幅:{width}, mode:{mode} ,エポック数:{num_epoch}, 学習率:{lr}")
for epoch in range(num_epoch):
    optimizer.zero_grad()
    pred = model(features)
    loss = criterion(pred, labels)
    loss.backward()
    optimizer.step()
print("訓練が完了しました。")

# 4. 近似用ラッパー関数
def f_approx(x, y):
    # モデルと同じデバイス（GPU）に入力を送る
    coords = torch.tensor(np.stack([np.array(x).ravel(), np.array(y).ravel()], axis=1), dtype=torch.float32).to(device)
    with torch.no_grad():
        # 推論結果を .cpu() で一度CPUに戻してからNumPy配列に変換する
        res = model(coords).cpu().numpy().reshape(np.shape(x))
    return res

# 5. 誤差解析
n_eval = 51
x_eval = np.linspace(0, 1, n_eval)
y_eval = np.linspace(0, 1, n_eval)
X_eval, Y_eval = np.meshgrid(x_eval, y_eval, indexing='ij')
f_eval = f(X_eval, Y_eval)
f_eval_approx = f_approx(X_eval, Y_eval)

abs_err = np.abs(f_eval - f_eval_approx)
rel_err = abs_err / np.max(np.abs(f_eval))
print(f"Max Relative Error: {np.max(rel_err):.6f}")

# 6. 可視化
fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X_eval, Y_eval, f_eval, cmap='viridis', alpha=0.8)
ax1.set_title("True Function")

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(X_eval, Y_eval, f_eval_approx, cmap='plasma', alpha=0.8)
ax2.set_title("NN Approximated (with Rational)")

fig_err = plt.figure(figsize=(7, 6))
ax3 = fig_err.add_subplot(1, 1, 1, projection='3d')
ax3.plot_surface(X_eval, Y_eval, rel_err, cmap='inferno', alpha=0.8)
ax3.set_title("Relative Error")

plt.show()